# model_04 - deneme 2 . ODUL KOLU

Kullanici karari, 17 Eylul 2026: *"tamam odul mekanizmasini model_03
uzerine kuracagiz"*, ve merdivenin bicimi icin: *"basit bir matematik
hesabiyla degil; sekil, yuva, beklenen cevaba yaklasma gibi biraz
kompleks kriterlerle verilmeli."*

Onceden kayit `belge/onkayit/model_04.md`.

### DUGME

```
odul_ac   False -> True     model_03'un UZERINE, ent_arama sorularinda
lr        1e-3  -> 1e-5     odul bir INCE AYAR asamasi
```

Diger her sey model_03 ile AYNI: veri, egitim havuzu, mimari, wd 0.5,
sabit LR, t_len 17, bolmeler, olcme izi d6751004648c.

Kol **sifirdan kosmuyor**: model_03/t0'in 20.000. adim anlik
goruntusunden basliyor (4. hucre, `BASLANGIC`).

> **ATFETME YAPILAMAZ.** Iki alan birden degisiyor. Ama `lr` serbest bir
> dugme DEGIL: 1e-3'te kosu modeli 30 adimda siliyor (asagida).

### NEYI SINIYOR

model_03 `comp`i 0.8350'ye cikardi ama `ent` yerinde saydi
(0.0417 -> 0.0423). Olculdu: `ent`te dogru cevap ORNEKLEME ALTINDA VAR,
argmax'a cikmiyor.

```
model_03, ent    argmax 0.0423   pass@8 0.0735   pass@256 0.2369
                 dogru cevabin ortanca sirasi 50 / 1060
```

Soru: **odul, orada duran ama argmax'a cikmayan cevabi one cikarabilir
mi?** Literaturun soyledigi tavan pass@256 = 0.2369 (2504.13837: odul
taban modelin dagiliminda olani one cikarir, YENISINI YARATMAZ).

### MERDIVEN

Bir cevap uc yuvadir. Once UC KAPI, sonra basamaklar. Bir kapi duserse
alt katmanlara BAKILMAZ.

```
KAPI 1   <YOK> SAGDA mi       cevaba BAKMAZ    duserse ZEMIN -0.50
KAPI 3   GERCEK bir varlik mi cevaba BAKMAZ    duserse ZEMIN -0.50

ozneden 1 ADIMDA ulasilir      KISAYOL          -0.30
gecerli varlik, YANLIS tip                      -0.20
dogru tip, menzil DISI                           0.00
2 ADIMDA ulasilir                                0.34
  + yuva2 (aile/tur) dogru                       0.67
  + yuva1 de dogru (TAM)                         1.00
```

!! KAPI 2 (jeton sayisi) KAPI DEGIL, BASAMAK -- 17 Eylul duzeltmesi.
Kapi oldugunda 2 adim menzilindeki varliklarin %55'i (dogru tipte
olmayanlar) zemine dusuyordu ve "menzilden rastgele sec", "tipten
rastgele sec"ten DAHA AZ aliyordu (0.0048 vs 0.1048) -- merdiven ters
donmustu. KAPI 1 ve 3 BICIM kapisi; KAPI 2 ICERIK.

**Pozitif basamaklar UYDURULMADI** -- aramanin ne kadar daraldigindan
turetildi:

```
basamak                  kalan aday   kumulatif bilgi   ODUL
baslangic                   1060         0.00 bit         -
E  gecerli + dogru jeton     574         1.23 bit       0.12
F  2 adim menzilinde         104         3.38 bit       0.34
G  + dogru aile / tur         11         6.78 bit       0.67
H  TAM                         1        10.05 bit       1.00
```

Yani odul = "cevabin bilgisinin ne kadarini daralttin". Tek keyfi karar
H = 1.00; o da olcek. ZEMIN ve KISAYOL cezasinin turetmesi YOK --
arizanin tanimindan geliyorlar, ikisi de ACIK DUGME.

`ZEMIN < KISAYOL` ZORUNLU: aksi halde bozuk cevap, kopruyu atlamaktan
karli olur ve model belirsizlikte sacmalamayi ogrenir.

### ORNEK -- ayni soruda sekiz dal

```
SORU   : Oya Kaya'in rakibinin komsusu ?
DOGRU  : Ipek Yildiz      KOPRU: Fatih Kaya      KISAYOL: Pinar Demir

cevap                    nerede takildi              ODUL
Ipek <YOK> Yildiz        KAPI 1 -- bosluk ortada    -0.50
Ipek Yildiz Lisesi       KAPI 2 -- 3 jeton          -0.50
Zeynep Yildiz            KAPI 3 -- boyle biri yok   -0.50
Pinar Demir              KISAYOL -- kopruyu atladi  -0.30
Ahmet Yilmaz             menzil disi                +0.12
Ayse Yilmaz              2 adimda, aile yanlis      +0.34
Hulya Yildiz             2 adimda + aile dogru      +0.67
Ipek Yildiz              TAM DOGRU                  +1.00
```

### ODUL HANGI SORULARDA KOSAR -- ent_arama

`ent` bir HUKUM bolmesi; uzerinde odul kosmak sinavi egitmek olur.
`ent_arama` proje tasariminda zaten "hukumde kullanilmaz" diye ayrilmis.
Ve OLCULDU:

```
zincir-basi varlik kumeleri
  ent_arama    53 varlik
  ent         159 varlik
  KESISIM       0          <- TAMAMEN AYRIK
```

Yani odul ent_arama varliklarini zincir basi YAPAR, ama `ent`
varliklari ne denetimli egitimde ne odul asamasinda zincir basi olur.
`ent` gercek bir TUTULMUS SINAV olarak kalir. `taban_04` bunu her
kosuda assert ediyor.

### OLCULEN: LR ZORUNLULUGU

Duman testi, 30 odul adimi, model_03'un agirliklarindan baslayarak:

```
   lr        one      seen      comp      ent
(baslangic) 0.9267   0.9133    0.7867   0.0433
  1e-3      0.0067   0.0067    0.0067   0.0033   <- TAM COKME
  1e-4      0.9233   0.8667    0.7267   0.0367
  1e-5      0.9233   0.9133    0.7867   0.0433   <- SECILEN
  1e-6      0.9267   0.9133    0.7867   0.0433
```

30 adimlik olcum; 20.000 adimda hasar BIRIKIR, yani bu tablo IYIMSER.
`one` ve `seen` on kosullari tam bu yuzden KORUMA olcusu.

### KARAR KURALI - son pencere

**BIRINCIL OLCU `ent`** -- `comp` bu kolda birincil DEGIL, model_03'te
zaten 0.8350 ve pass@8 0.9087; odulun orada yapacak isi yok.

```
ent                HUKUM                     (model_03: 0.0423)
>= 0.2369          ORNEKLEME TAVANINI (pass@256) tutturdu
0.10 - 0.2369      ODUL ARGMAX'A TASIDI -- kolun iddiasi DOGRULANDI
0.05 - 0.10        kismi
<= 0.05            odul `ent`i acmadi

ON KOSUL (KORUMA)  one >= 0.98  VE  seen >= 0.95  VE  comp >= 0.50
                   gecmezse `ent` YORUMLANMAZ: taban beceri eridiyse
                   kazanc degil TAKASTIR

BIRIM TESTI        ent_yok_kisayol == 0.000   <- KIRMIZI CIZGI
```

`ent_kisayol` ve `od_*` sutunlari RAPORLANIR, HUKUM VERMEZ.

> **SAYISAL TAHMIN YAZILMIYOR** (kullanici karari, 15 Eylul).

### Beklenen kilit ciktisi

```
model_04/test_04.py    150 gecti, 0 BOZUK
  ok  odul ACIK -- model_04'un TANIMI
  ok  odul bolmesi ent_arama
  ok  ent_arama ile ent ZINCIR-BASI varliklari AYRIK
  ok  merdiven: sekiz dalin sekizi de beklenen puani verdi
  ok  avantaj: SABIT terim TAM OLARAK iptal oluyor
  ok  avantaj: VARYANSSIZ grup -> gradyan SIFIR
  ok  OdulAyar TERS merdiveni REDDEDIYOR
  ok  lr 1e-5 -- ODUL ASAMASI degeri
  ok  Ayar'a YALNIZ odul alanlari eklendi
```

---

**Sira:** 0 -> 1 -> 2 -> 3 -> 4 ile baslat. Kosu surerken **5 ve 6**.
Bitince **7** (`pencere_04`) ve **9** (`sor_04`).


In [ ]:
# 0 MODEL ADI VE YOLLAR  |  CPU  |  tekrar: GUVENLI
MODEL = "model_04"            # <-- DEGISTIRILECEK TEK SATIR

assert MODEL, ("MODEL bos. Bu SABLON -- kopyala, adini modelin adi yap "
               "(model_a.ipynb) ve bu satiri doldur.")
import time
DEPO  = "https://github.com/sekerahmet/sekerai.git"
KOD   = "/content/kod"
EV    = f"/content/drive/MyDrive/{MODEL}"     # TEPE KLASOR = MODELIN ADI
# LOG ADI BURADA URETILMEZ -- 4. hucre her BASLATMADA kendi damgasini
# basar. Sebep olculdu (15 Eylul): log adi burada uretilince, bu hucreyi
# yeniden calistirmadan ikinci bir kosu baslatmak ONCEKI kosunun logunu
# "w" ile ACIP SIFIRLIYOR. Fiilen oldu: 20.000'lik kosunun logu, 40.000'lik
# kosu baslayinca silindi. Damga BASLATAN hucrede uretilirse imkansiz.
print(MODEL, "->", EV)

In [ ]:
# 1 GPU VAR MI, BOS MU  |  GPU'yu SORAR, kullanmaz  |  tekrar: GUVENLI
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_p = torch.cuda.get_device_properties(0)
_bos = torch.cuda.mem_get_info()[0] / 1e9
print(f"{_p.name}   toplam {_p.total_memory/1e9:.1f} GB   bos {_bos:.1f} GB")
assert _bos > 3.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"

In [ ]:
# 2 DRIVE  |  CPU  |  tekrar: GUVENLI (yalniz <model>/log/ acar)
from google.colab import drive
import os
drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), \
    "drive.mount CALISMADI -- /content/drive gercek bir baglanti degil."
os.makedirs(f"{EV}/log", exist_ok=True)
with open(f"{EV}/.yazma_denemesi", "w") as f:
    f.write("ok")
os.remove(f"{EV}/.yazma_denemesi")
print("hazir ve YAZILABILIR:", EV)
print("  var olan tohum klasorleri:",
      sorted(d for d in os.listdir(EV) if d.startswith("t")) or "(yok)")

In [ ]:
# 3 KODU GITHUB'DAN CEK + KILIT TESTI  |  CPU  |  tekrar: KOSU YOKKEN (ilk isi rm -rf /content/kod)
import subprocess, os, glob, importlib, sys
subprocess.run(["rm", "-rf", KOD], check=True)
subprocess.run(["git", "clone", "--depth", "1", DEPO, KOD], check=True)
COMMIT = subprocess.run(["git", "-C", KOD, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("commit:", COMMIT, "|",
      subprocess.run(["git", "-C", KOD, "log", "-1", "--format=%s"],
                     capture_output=True, text=True).stdout.strip())

# Her modelin AILE klasoru var: deneme2/model_a/{model_a.py, pencere_a.py,
# model_a.ipynb}. Klasor adini isimden TURETMIYORUZ, dosyayi ARIYORUZ --
# model_a1 gibi varyasyonlar da ayni aile klasorunde durur.
# !! model_04 ARANMAZ: kendi klasoru BELLI. Kullanici karari,
# 16 Eylul -- "model_04 diger hicbir model ile ayni seyi kullanmamali".
# Paylasilan sablon `deneme2/*/<MODEL>.py` glob'u yapiyordu; model_04
# artik o aramaya girmiyor.
_aday = glob.glob(f"{KOD}/deneme2/model_04/{MODEL}.py")
assert len(_aday) == 1, f"{MODEL}.py tam bir kez bulunmali, bulunan: {_aday}"
AILE = os.path.dirname(_aday[0])
_pen = glob.glob(f"{AILE}/pencere_*.py")
assert len(_pen) == 1, f"ailede tam bir pencere_*.py olmali: {_pen}"
PENCERE = _pen[0]
print("aile:", AILE, "| olcum:", os.path.basename(PENCERE))

# --- IMPORT ONBELLEGINI TEMIZLE -- BU HUCRENIN EN SESSIZ TUZAGI ----------
# `rm -rf` + yeniden klon KODU tazeler ama `sys.modules` ESKI modul
# nesnesini tutar. Ayni cekirdekte MODEL degistirip bu hucreyi yeniden
# kosarsan, yeni kodu klonlamis ama ESKI modulu kullaniyor olursun.
# OLCULDU (15 Eylul): model_a4'ten model_a5'e gecerken
#   "AssertionError: Ayar'da boyle alan yok: {'ort_bas'}"
# cikti -- cunku model_a hala onceki klondan gelen, `ort_bas`i olmayan
# nesneydi. Daha sinsi hali: alan adlari tutarsa hata VERMEZ ve kosu
# ESKI KODLA baslar; kunyedeki commit ise YENIYI gosterir.
_atilan = [n for n, m in list(sys.modules.items())
           if getattr(m, "__file__", None) and str(m.__file__).startswith(KOD)]
for n in _atilan:
    del sys.modules[n]
importlib.invalidate_caches()
if _atilan:
    print("import onbellegi temizlendi:", sorted(_atilan))

# --- KILIT: test_04.py ---------------------------------------------------
# model_04 KENDI motoruna (taban_04.py) ve KENDI verisine (veri_04.py)
# sahip -- ikisi de model_a / veri_okul KOPYASI. `test_04.py` dort sey
# tutuyor: (0) BAGIMSIZLIK, model_04/*.py disariya import ETMIYOR;
# (1) MIMARI; (2) VERI, graf veri_okul4'unkiyle BIREBIR; (3) MOTOR,
# egitim havuzu ve olcme izi model_a ile AYNI.
# Duserse egitim BASLAMAMALI -- sayilar baska bir tabloda okunur.
#
# CIKTI YAKALANIR ve BASILIR: Colab alt surec stdout'unu hucreye
# aktarmiyor; "cikti yok" ile "test kosmadi" ayrimi sansa birakilmaz.
# AYRINTI=1 -> gecen kontroller de basilir (dugme tablosu gorunur olsun).
_t = [x for x in glob.glob(f"{AILE}/test_*.py")]
if _t:
    print()
    print("=" * 72)
    _r = subprocess.run([sys.executable, _t[0]], cwd=AILE,
                        capture_output=True, text=True,
                        env={**os.environ, "AYRINTI": "1",
                             "KOSU_KOK": EV.rsplit("/", 1)[0]})
    print(_r.stdout.rstrip() or "(cikti YOK -- test kosmamis olabilir!)")
    if _r.stderr.strip():
        print("stderr:", _r.stderr.rstrip()[-2000:])
    print("=" * 72)
    assert _r.returncode == 0, (
        f"{os.path.basename(_t[0])} DUSTU (cikis {_r.returncode}). Taban "
        f"degismis olabilir ve KAYITLI sonuclar ona dayaniyor. EGITIM BASLATMA.")
    print()

sys.path.insert(0, AILE)
# !! UST KLASOR EKLENMIYOR: veri modulu de (veri_04.py) AILE icinde.
# model_04 `deneme2/` kokundeki hicbir seyi gormez.
M_04 = importlib.import_module(MODEL)
M = M_04
# `M_04.M` MOTOR (taban_04). Paylasilan sablonda `M` TABANI
# gosteriyordu (model_a); burada `M` kolun KENDISI, motor ise
# `M_04.M`. Miras denetimi taban sinifi ORADAN alir.
MOTOR = M_04.M
assert M.AYAR.ad == MODEL, f"AYAR.ad {M.AYAR.ad!r} != dosya adi {MODEL!r}"
# Yuklenen modul GERCEKTEN yeni klondan mi geldi?
assert M.__file__.startswith(KOD), f"{MODEL} {KOD} disindan geldi: {M.__file__}"
# BU KOLUN SARTLARI. Hicbiri DEVRALINMIYOR: `ayar_04.py` her alani
# `Ayar()` varsayilaninin uzerine tek tek, gerekcesiyle yaziyor.
# --- MIMARI: bu kolun TANIMI --------------------------------------
assert M.AYAR.dongu == 1,              "DONGU YOK"
assert M.AYAR.l == 8,                  "8 AYRI katman"
assert M.AYAR.dar_alfa == 0.0,         "Phi DARBOGAZI YOK"
assert M.AYAR.dar_kapi is False,       "ogrenilen GECIT YOK"
assert M.AYAR.dff == 704,              "SwiGLU d_ff = 8/3*d"
assert M.AYAR.d // M.AYAR.nh == 64,    "head_dim 64"
# KUSUR (16 Eylul, defterin ILK kosusu): burada `M.Model` yaziyordu
# ve `M` = model_04 modulu; onda `Model` YOK -> AttributeError.
# Sablondan devralinmisti, defter hic kosulmadigi icin gorulmedi.
assert not issubclass(M_04.ModelSade, MOTOR.Model), \
    "ModelSade taban_04.Model den MIRAS ALMAMALI"
# ======================================================================
# BURADAN ASAGISI MIMARI DEGIL. Uc AYRI kategori, karistirilmasin:
#
#   A) DILIN KENDISI (korpus)  -- hiperparametre DEGIL. "ek_kip=tr"
#      demek "bu dil ekli bir dil" demek; "bicim=3" demek "ayni olgu
#      uc yuzey biciminde geciyor" demek. Modelin secimi degil,
#      METNIN ozelligi.
#   B) SINAV BOLMELERI          -- olcumun tanimi, modele ait DEGIL.
#   C) PROJEYE OZGU VERI EKI    -- BEST PRACTICE DEGIL. Tek kalem:
#      identity bridge (ident_frac / ident_kip). model_b13'te
#      arXiv 2509.24653'ten alindi. Burada DURUYOR cunku egitim havuzu
#      model_b15 ile BIT DUZEYINDE ayni olmali; olmazsa kol "mimari mi
#      veri mi" sorusunu AYIRAMAZ. Kaldirilirsa kol AYRI bir soru sorar
#      (onkayit model_04.md 2).
#
# Optimizasyon (wd/cosine/isinma/lr/betas) YUKARIDA degil ASAGIDA ve
# hepsi STANDART TARIF -- proje kisiti DEGIL.
# ======================================================================
# --- A) DILIN KENDISI -------------------------------------------------
# --- BU KOLUN TEK DUGMESI ------------------------------------------
# --- BU KOLUN TANIMI: ODUL ---------------------------------------
assert M.AYAR.odul_ac is True,         "ODUL ACIK -- kapaliysa bu model_03"
assert M.AYAR.odul_bolme == "ent_arama", (
    "odul `ent` uzerinde KOSAMAZ -- o bir HUKUM bolmesi")
assert M.AYAR.odul_denetimli is False, "denetimli kayip KAPALI"
assert M.AYAR.odul_kl == 0.0,          "KL KAPALI -- atfetme bozulmasin"
_M = (M.AYAR.odul_zemin, M.AYAR.odul_kisayol, M.AYAR.odul_tip,
      M.AYAR.odul_e, M.AYAR.odul_f, M.AYAR.odul_g_aile, M.AYAR.odul_h)
assert all(x < y for x, y in zip(_M, _M[1:])), f"MERDIVEN BOZUK {_M}"
assert M.AYAR.odul_zemin < M.AYAR.odul_kisayol, (
    "ZEMIN kisayolun ALTINDA olmali, yoksa model SACMALAMAYI ogrenir")
assert M.AYAR.odul_e == 0.00, "E menzil disi -- agirlik MENZILE kaydirildi"
assert M.AYAR.odul_tip < M.AYAR.odul_e, "YANLIS TIP, E'nin ALTINDA"
print("merdiven:", _M, " G", M.AYAR.odul_g, " T", M.AYAR.odul_sicaklik)
# --- model_03ten DEVRALINAN RECETE (DEGISMEDI) -------------------
assert M.AYAR.wd == 0.5,               "model_a3 RECETESI -- DEVRALINDI"
assert M.AYAR.sabit_lr is True,        "LR SABIT -- recetenin ikinci yarisi"
assert not hasattr(M.AYAR, "dusun_gecis"), "model_04te DONGU YOK"
assert M.AYAR.veri_ad == "veri_04",    "model_04 KENDI veri modulu"
import veri_04 as _V03
assert _V03.graf_izi(_V03.kur(0)) == _V03.IZ, "graf KAYMIS"
print(f"veri_04  graf izi {_V03.IZ}  (icerik veri_okul4, 1060 varlik)")
assert M.AYAR.jeton_ad == "tam",       "varlik = JETON DIZISI (3 yuva)"
assert M.AYAR.ek_kip == "tr",          "dil EKLI: '  <NIN>  <SI>  <DIR>"
assert M.AYAR.bicim == 3,              "ayni olgu UC yuzey biciminde"
assert M.AYAR.t_len == 17,             f"t_len DEGISMEDI -- SINAV AYNI: {M.AYAR.t_len}"
assert M.AYAR.belge_pay == 0.0,        "BELGE satiri YOK (ek_kip ile kurulmadi)"

# --- B) SINAV BOLMELERI -- olcumun tanimi -----------------------------
assert M.AYAR.ood_pay == 0.05,         f"ood bolmesi: {M.AYAR.ood_pay}"
assert M.AYAR.kopru_kayip == 0.0,      "YARDIMCI KAYIP YOK (bu MIMARI karari)"

# --- C) PROJEYE OZGU VERI EKI -- BEST PRACTICE DEGIL ------------------
# Kullanici, 16 Eylul: "biz her seyi sifirdan yaptik, ben kisit
# vermedim, best practice dedim." Dogru -- ve bu IKI SATIR o tarifin
# parcasi DEGIL. Egitim havuzu b15 ile ayni kalsin diye duruyor.
assert M.AYAR.ident_frac == 0.2,       "identity bridge -- PROJE EKI, b15 ile AYNI"
assert M.AYAR.ident_kip == "q1",       "identity bridge -- PROJE EKI, b15 ile AYNI"

# --- OPTIMIZASYON: STANDART TARIFIN KENDISI ---------------------------
# Bunlar proje kisiti DEGIL. Dordu de ayni sayilari kullaniyor:
#   nanoGPT   GPT-3   Llama   Pythia
# !! lr STANDART TARIFTEN CIKTI: 1e-3 SIFIRDAN egitimin degeri.
# Odul bir INCE AYAR asamasi ve 1e-3 modeli 30 adimda SILIYOR
# (olculdu 17 Eylul: one 0.9267 -> 0.0067).
assert M.AYAR.lr == 1e-5,              "ODUL ASAMASI LR'i"
assert M.AYAR.isinma == 2000,          "nanoGPT warmup_iters=2000, Llama 2000"
assert M.AYAR.betas == (0.9, 0.95),    "nanoGPT/GPT-3/Llama/Pythia -- 0.999 DEGIL"
assert M.AYAR.tam_kayip is True,       "butun pozisyonlarda next-token: standart LM"
# PROJEYE OZGU OPTIMIZASYON NUMARALARI -- HEPSI KAPALI
assert M.AYAR.ort_bas == 0,            "LOOKAHEAD KAPALI -- hicbir tarifte YOK"
assert not (M.AYAR.dar_sert or M.AYAR.dar_sdpa), "darbogaz zaten YOK"
print("mimari: ModelSade  8 katman  RoPE  SwiGLU  bagli gomme  bias YOK")
SOR = os.path.join(AILE, "sor_04.py")  # 9. hucre kullanir
for _g in ("AYAR", "egit", "fark_bas"):
    assert hasattr(M, _g), f"{MODEL}'de {_g} YOK -- kos_04.py duser"
print("ayar:", M.AYAR)

In [ ]:
# 4 BASLAT  |  GPU (alt surec)  |  tekrar: HAYIR -- yeni kosu baslatir
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
# GPU kullanacak her hucre, ONCE kendi icinde GPU'yu sorar. Ayri bir
# "GPU var mi" hucresi HUCRE SIRASINA bagli bir kuraldir; icerideki
# kontrol degildir. Olculdu (15 Eylul): Colab cekirdegi kosu sirasinda
# oldu, GPU ve Drive durumu gitti; ayri kontrol hucresi vardi ama
# calistirilmamisti.
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------

TOHUMLAR = [0]             # ONCE TEK TOHUM.
# !! BU KOLUN OLCUTU `comp` (ent DEGIL) -- onkayit model_04.md.
# Sonuc OLUMLU cikarsa (comp yukseldiyse) tek tohum YETER.
# OLUMSUZ cikarsa bu satiri [1, 2] yapip tekrar kos -- t0 klasoru
# DOKUNULMAZ, yeni kosular t1/ ve t2/'ye yazar. Kod degismez.
# Neden onemli: grokking tohuma bagli. Tek tohumda comp yukselmezse
# "konfigurasyon yanlis" ile "bu baslangic sanssiz" AYRILAMAZ.
# VERI tohumu AYRI (ayar.veri_tohum=0) -- butun tohumlar AYNI veriyi gorur.

# !! ADIM buyutup SURDUR=True yapmak UZATMADIR, sifirdan kosu DEGIL.
#    Ama cosine ufku `ayar.adim`dan turedigi icin LR GERI FIRLAR:
#    model_b14'te 20.000 ufkunda lr/10 iken 60.000 ufkunda 7,6 KAT.
#    Uzatilmis kosu, temiz bir uzun kosu DEGILDIR.
ADIM   = None    # None = ayar_04.py'deki ILK SINIR (20.000).
#                  Uzatmak KULLANICI karari (CLAUDE.md kural 1).
SURDUR = False   # TAZE kosu -- surdurme DEGIL.

# !! BU KOL SIFIRDAN KOSMAZ. Odul, model_03'un EGITILMIS agirliklarinin
# UZERINE biniyor (kullanici karari, 17 Eylul). Sifirdan odul kosmak
# BASKA bir soru sorar ve cevabi zaten olculmus: odul, taban modelin
# ornekleme dagiliminda ZATEN olani one cikarir, yenisini YARATMAZ
# (2504.13837). `kos_04.py` --baslangic YOKSA kosuyu REDDEDER.
_M03 = os.path.dirname(EV) + "/model_03/t0/snap"
# KLASOR veriliyor, tek dosya DEGIL -> kos_04 son 5 anlik goruntunun
# PENCERE ORTALAMASINI alir. Olculdu (17 Eylul):
#   model_03 egri 20.000    one 0.9153  seen 0.9210  comp 0.8113
#   model_03 PENCERE 12-20k one 0.9857  seen 0.9960  comp 0.8350
# Tek goruntuden baslamak kosuyu ON KOSULUN ALTINDAN baslatirdi
# (one 0.9153 < 0.98) ve sonuc YORUMLANAMAZDI. Ayrica bu kolun butun
# kiyas sayilari PENCERE modelinden olculdu.
BASLANGIC = _M03
assert os.path.isdir(BASLANGIC), "model_03 snap klasoru YOK: " + BASLANGIC
assert len([x for x in os.listdir(BASLANGIC) if x.endswith(".pt")]) >= 5, (
    "pencere icin en az 5 anlik goruntu gerek: " + BASLANGIC)
print("baslangic: PENCERE ORTALAMASI ->", BASLANGIC)
USTUNE = False   # t0 BOS (yeni kol). True = dolu klasoru
#                  t<N>_eski_<zaman>/'a TASI (silmez)

if "p" in globals() and p.poll() is None:
    raise SystemExit(f"ZATEN KOSUYOR (PID {p.pid}). Once 'Durdurmak' hucresi.")

# KENDI kosucusu: model_04/kos_04.py. `--model` YOK -- bu betik
# yalnizca model_04'i baslatir (paylasilan kos.py aile klasoru ARIYORDU).
_arg = [sys.executable, "-u", f"{KOD}/deneme2/model_04/kos_04.py",
        "--ev", EV, "--commit", COMMIT,
        "--tohum", *[str(t) for t in TOHUMLAR]]
if ADIM:
    _arg += ["--adim", str(ADIM)]
if SURDUR:
    _arg += ["--surdur"]
if USTUNE:
    _arg += ["--ustune"]
if not SURDUR:
    _arg += ["--baslangic", BASLANGIC]
# LOG ADI HER BASLATMADA YENI: bu hucre iki kez calisirsa iki AYRI log
# olur, oncekinin uzerine YAZILMAZ.
LOG = f"{EV}/log/kos_{time.strftime('%Y%m%d_%H%M%S')}.txt"
p = subprocess.Popen(_arg, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("PID", p.pid, " tohum", TOHUMLAR, " -> log:", LOG)
print("Dolu bir tohum klasoru varsa kosu REDDEDILIR -- hicbir sey ezilmez.")
print("Ayni tohumu bilerek tekrar kosmak icin --ustune; o da SILMEZ,")
print("eskisini t<N>_eski_<zaman>/ diye yan klasore TASIR.")

In [ ]:
# 5 ILERLEME (ham log)  |  CPU  |  tekrar: GUVENLI (log geriden gelebilir)
import glob, subprocess
# LOG degiskenine DEGIL, klasordeki EN YENI log'a bak.
_l = sorted(glob.glob(f"{EV}/log/kos_*.txt"))
assert _l, f"log yok: {EV}/log/"

# `p` CEKIRDEK YENIDEN BASLAYINCA KAYBOLUR -- ve tam o an bu hucreye
# ihtiyac duyulur. Olculdu (15 Eylul): Colab cekirdegi oldu, bu hucre
# `NameError: name 'p' is not defined` verdi, koşunun yasayip yasamadigi
# ogrenilemedi. Artik `p` yoksa SUREC TABLOSUNA bakiyor.
if "p" in globals():
    _d = p.poll()
    print("KOSUYOR" if _d is None else f"BITTI/OLDU (cikis kodu {_d})",
          "| PID", p.pid)
else:
    _ps = subprocess.run(
        ["bash", "-lc", "ps -eo pid,etime,cmd | grep kos_04.py | grep -v grep"],
        capture_output=True, text=True).stdout.strip()
    print("!! `p` YOK -- cekirdek yeniden baslamis.")
    if _ps:
        print("   ama SUREC YASIYOR:\n   " + _ps)
    else:
        print("   ve kos_04.py sureci de YOK -> kosu OLDU.")
        print("   Drive'i yeniden bagla (2. hucre), 3'u kos, sonra 4. hucrede")
        print("   SURDUR = True ile KALDIGI YERDEN devam ettir.")
print("log:", _l[-1].split("/")[-1], f"({len(_l)} log dosyasi var)")
print("-" * 78)
print(open(_l[-1]).read()[-4000:])

In [ ]:
# 6 RAPOR (canli durum)  |  CPU  |  tekrar: GUVENLI
import json, glob, os, statistics

# KIYAS TABANI: model_03 -- bu kolun BASLADIGI YER. Ayni mimari, ayni
# veri, ayni sinav (iz d6751004648c), ayni havuz, ayni wd/sabit_lr.
# FARK: odul ACIK ve lr 1e-3 -> 1e-5. Iki alan, ATFETME YAPILAMAZ.
M03 = dict(one=0.9857, seen=0.9960, comp=0.8350, ood=0.0190,
           ent=0.0423, ent_yok=0.0414)      # model_03 PENCERE 12-20k
M03_KSY = 0.0903
# ODULUN TAVANI -- olculdu (OLCULENLER.md 1e). RLVR literaturu (2504.13837)
# "odul taban modelin ornekleme dagiliminda olani one cikarir, yenisini
# YARATMAZ" diyor. model_03'un ent pass@256'si bu:
ENT_TAVAN = 0.2369
M03_MS = 77.0                               # model_03 OLCULDU (t_len 17)

_TUM = (("adim", "adim"), ("kayip", "kayip"), ("one", "one"),
        ("seen", "seen"), ("comp", "comp"), ("ood", "ood"), ("ent", "ent"),
        ("ent_yok", "ent_yok"), ("kayip_ana", "kayip_ana"),
        ("odul", "odul"), ("od_kapi", "kapi"),
        ("od_ornek_menzil", "menzil"), ("od_tam", "od_tam"),
        ("od_ornek_kisayol", "od_ksy"), ("od_avantajli_soru", "avantajli"),
        ("ent_kisayol", "ent_ksy"), ("ent_yok_kisayol", "yok_ksy"))

for kl in sorted(k for k in glob.glob(f"{EV}/t*") if os.path.isdir(k)):
    eg = glob.glob(f"{kl}/egri_*.json")
    if not eg:
        print(f"{os.path.basename(kl)}: egri YOK"); continue
    ky = glob.glob(f"{kl}/kosu_t*.json")
    k = json.load(open(ky[0])) if ky else {}
    e = json.load(open(eg[0]))
    _var = set().union(*(set(r) for r in e))
    SUT = tuple(x for x in _TUM if x[0] in _var)
    _atlanan = [c for c in _var
                if c.startswith(("one", "seen", "comp", "ood", "ent", "odul",
                                 "od_"))
                and c not in {x[0] for x in SUT}]
    assert not _atlanan, f"OLCULUYOR AMA BASILMIYOR: {_atlanan}"
    _iz = k.get("olcme_izi", "?")
    assert _iz == "d6751004648c", (
        f"olcme izi {_iz} != d6751004648c -- model_03 KIYASI GECERSIZ")
    print("=" * 96)
    print(f"{os.path.basename(kl)}   {k.get('ad','?')}   durum {k.get('durum','?')}"
          f"   commit {k.get('commit','?')}   {k.get('gpu','')}")
    print("   " + "".join(f"{b:>10}" for _, b in SUT) + f"{'dk':>6}")
    for r in e:
        hcr = []
        for c, _ in SUT:
            x = r.get(c)
            hcr.append(f"{x:>10d}" if c == "adim" else
                       (f"{'---':>10}" if x is None else f"{x:>10.4f}"))
        print("   " + "".join(hcr) + f"{r['sn']/60:>6.0f}")

    s = e[-1]
    print("   " + "-" * 93)
    # !! BU BLOK EGRI OKUMASI -- son olcum noktasinin TEK anlik goruntusu.
    # HUKUM PENCEREYLE verilir (7. hucre, pencere_04). Ikisi AYRISABILIR:
    # model_03'te egri one 0.9153 ile kapida KALDI, pencere 0.9857 ile
    # GECTI. CLAUDE.md kural 3.
    print("   [EGRI okumasi -- HUKUM DEGIL. Hukum: 7. hucre, pencere_04]")
    print(f"   {'GECTI ' if s.get('one',0) >= 0.98 else '!! KALDI'}"
          f"  {'SAGLIK-1HOP':<14} one >= 0.98      <- ON KOSUL (KORUMA)")
    print(f"   {'GECTI ' if s.get('seen',0) >= 0.95 else '!! KALDI'}"
          f"  {'SAGLIK-EZBER':<14} seen >= 0.95     <- ON KOSUL (KORUMA)")
    print(f"   {'GECTI ' if s.get('comp',0) >= 0.50 else '!! KALDI'}"
          f"  {'OLGUNLUK':<14} comp >= 0.50     <- ON KOSUL (KORUMA)")
    print(f"   {'GECTI ' if abs(s.get('ent_yok_kisayol',1)) < 1e-9 else '!! KALDI'}"
          f"  {'BIRIM TESTI':<14} ent_yok_kisayol == 0   <- KIRMIZI CIZGI")

    print("   " + "-" * 93)
    d = [(b["sn"] - a["sn"]) / (b["adim"] - a["adim"]) * 1000
         for a, b in zip(e, e[1:]) if b["sn"] > a["sn"] and b["adim"] > a["adim"]]
    if d:
        ms = statistics.median(d)
        print(f"   HIZ ortanca {ms:.1f} ms/adim   (model_03 {M03_MS})"
              f"   {(ms/M03_MS-1)*100:+.1f}%")
    print(f"   {'olcu':<9}{'model_03':>11}{'model_04':>10}{'fark':>10}")
    for c in ("one", "seen", "comp", "ood", "ent", "ent_yok"):
        if c in s:
            print(f"   {c:<9}{M03[c]:>11.4f}{s[c]:>10.4f}{s[c]-M03[c]:>+10.4f}")
    print("   model_03 sutunu BU KOLUN BASLADIGI YER (pencere 12-20k).")
    print("   Fark = odul + lr 1e-5'in birlikte yaptigi. AYRILAMAZ.")
    if "ent" in s:
        _e = s["ent"]
        print("   BIRINCIL (ent -- onkayit model_04.md 7): "
              + ("ORNEKLEME TAVANINI TUTTURDU" if _e >= ENT_TAVAN else
                 "ODUL ARGMAX'A TASIDI -- kolun iddiasi DOGRULANDI"
                 if _e >= 0.10 else
                 "0.05-0.10 kismi" if _e > 0.05 else
                 "<=0.05 odul `ent`i ACMADI"))
        print(f"      model_03 ent {M03['ent']:.4f} -> model_04 {_e:.4f}"
              f"   tavan (pass@256) {ENT_TAVAN:.4f}")
        print("      ON KOSUL: one VE seen VE comp. Gecmezse ent YORUMLANMAZ")
        print("      -- taban beceri eridiyse bu KAZANC degil TAKASTIR.")
    if "ent_kisayol" in s:
        print(f"   KISAYOL ent_ksy {s['ent_kisayol']:.4f}  (model_03 {M03_KSY:.4f})"
              f"   fark {s['ent_kisayol']-M03_KSY:+.4f}")
        print("      RAPORLANIR, HUKUM VERMEZ. Ama YUKSELDIYSE odul kisayolu")
        print("      arka kapidan geri getirmis olabilir -- birim testine bak.")
    if "odul" in s:
        print(f"   ODUL {s['odul']:.4f}   kapi {s.get('od_kapi',0):.4f}"
              f"   menzil {s.get('od_ornek_menzil',0):.4f}"
              f"   TAM {s.get('od_tam',0):.4f}"
              f"   avantajli soru {s.get('od_avantajli_soru',0):.4f}")
        print("      Bunlar ODUL BOLMESINDE (ent_arama) olculuyor -- HUKUM")
        print("      DEGIL. Hukum tutulmus `ent` bolmesinde verilir.")
    if "kayip_ana" in s and s["kayip_ana"] is None:
        print("   kayip_ana '---': odul kipinde denetimli hat HIC kosmadi,")
        print("      bu sutun TANIMSIZ. 'kayip sifira dustu' diye OKUNMAZ.")


In [ ]:
# 7 pencere_04 -- BIRINCIL OKUMA  |  GPU  |  tekrar: GUVENLI, ama ANCAK KOSU BITINCE
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
# GPU kullanacak her hucre, ONCE kendi icinde GPU'yu sorar. Ayri bir
# "GPU var mi" hucresi HUCRE SIRASINA bagli bir kuraldir; icerideki
# kontrol degildir. Olculdu (15 Eylul): Colab cekirdegi kosu sirasinda
# oldu, GPU ve Drive durumu gitti; ayri kontrol hucresi vardi ama
# calistirilmamisti.
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------

# BIRINCIL OKUMA. GPU kullanir (yukaridaki nota bak).
# PENCERE hucre 3'te ailenin icinden bulundu -- yolu elle yazmiyoruz.
TOHUM = TOHUMLAR[0]
!python {PENCERE} {EV}/t{TOHUM} --genislik 5

In [ ]:
# 8 DURDUR  |  CPU  |  tekrar: KOSUYU OLDURUR -- bastaki # bilerek duruyor
# Anlik goruntuler Drive'da kalir; surdurme paketi her olcum
# noktasinda yazilir, 4. hucrede SURDUR=True ile devam edilir.
# p.kill()

In [ ]:
# 9 sor_04 -- KENDI SORUNU SOR  |  GPU  |  tekrar: GUVENLI (OLCU DEGIL)
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK"
assert torch.cuda.mem_get_info()[0] / 1e9 > 2.0, "GPU dolu"
# ------------------------------------------------------------------------
# KENDI SORUNU SOR -- TURKCE. Listeye istedigin kadar satir ekle.
# Turkce harf SART DEGIL: "kardesi" de olur "kardesi" de (cozumleyici
# 1161 yazimi taniyor, hicbir ikisi cakismiyor).
# Ciplak yazim da calisir: "Ahmet Yilmaz anne kardes".
# !! BU BIR OLCU DEGIL -- elle sorulan sorular SECILMIS sorulardir.
SORULAR = [
    "Ahmet Yilmaz'in annesi",                    # 1-hop
    "Ahmet Yilmaz'in annesinin kardesi",         # 2-hop
    "Ahmet Yilmaz'in kardesinin okulu",          # 2-hop, cevap OKUL
    "Adana Lisesi'nin muduru",                   # 1-hop, OKUL -> KISI
    "Matematik'in hocasi",                       # 1-hop, DERS -> KISI
    "Adana'nin komsusunun valisi",               # 2-hop, SEHIR zinciri
]
_q = " ".join(f'--soru "{s}"' for s in SORULAR)
!python {SOR} {EV}/t{TOHUMLAR[0]} --genislik 5 {_q}

In [ ]:
# 10 asama1_04 --sonda -- KOPRU SONDASI  |  GPU  |  tekrar: GUVENLI
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------
# ASAMA-1 TESHISI -- model_04'da kopru hidden state'e girdi mi?
# !! BU KOLUN EK OKUMASI BURADA: Physics 3.1'in iddiasi
# 'augmentation olmadan bilgi EZBERLENIR ama DOGRUSAL KODLANMAZ'.
# Sonda slot 0 'bilgi VAR' derse comp acilmasa bile bu bir bulgu.
# arXiv 2505.17923 AYNI probe'u AYNI pozisyonda yapmis ve kopruyu
# BULMUS: 'the hidden representation of the last input token
# encodes information about all necessary bridge entities'.
# model_b13'te ayni pozisyonda 0.0275 cikmisti.
# MERDIVEN: model_b6 -> b9 -> model_b10 (TABAN) -> b13 -> b8.
#   model_b9 comp 0.0250 | model_b6 0.0442 | model_b8 TAVAN 0.9997
# --sonda : kopru DOGRUSAL okunabiliyor mu. Taban max(en_sik, KOPYA) --
#           kopru cogu zaman soru varligiyla ayni aileden, soyad GIRDIDE
#           duruyor (comp'ta kopya 0.4450). Bu duzeltilmeden slot1
#           yanlislikla "BILGI VAR" cikiyordu.

import os
ASAMA1 = os.path.join(AILE, "asama1_04.py")
assert os.path.exists(ASAMA1), ASAMA1
!python {ASAMA1} {EV}/t{TOHUMLAR[0]} --genislik 5 --sonda --birim --bolme comp,ent,ood,seen

In [ ]:
# 11 tani_04 -- ARIZA SEKLI  |  GPU  |  tekrar: GUVENLI
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------
# AYRISTIRMA: model YANLIS cevap verirken NE diyor?
#   KISAYOL (r2'yi dogrudan soru varligina uygulamis)
#   KOPRU   (ara varligi yazmis, ikinci hop'u yapmamis)
#   VARLIK_DEGIL / ILGISIZ / ...
# Ayrica: 1.hop tek basina, 2.hop tek basina, IKISI BIRDEN -> KAYIP.
# Bu bir HUKUM olcusu DEGIL, arizanin SEKLINI gosterir.
import os
TANI = os.path.join(AILE, "tani_04.py")
assert os.path.exists(TANI), TANI
!python {TANI} {EV}/t{TOHUMLAR[0]} --genislik 5

In [ ]:
# 12 tani_b -- model_b15 TABAN KIYASI  |  GPU  |  tekrar: GUVENLI
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------
# TABAN KIYASI: model_04'in ariza SEKLINI model_b15 ile kiyasla.
# !! AYNI SINAV KUMESI (iz d6751004648c) -- sayilar DOGRUDAN
#    kiyaslanabilir. Degisen YALNIZ mimari.
# model_b15 bir ModelB oldugu icin onu `tani_b.py` ile okuyoruz.
# Bu kolun ariza SEKLI model_b15'ten FARKLI mi, yoksa ayni mi?
# Ayniysa "mimari hicbir seyi degistirmedi" DAHA GUCLU soylenir.
# EGITIM YOK, yalniz okuma -- model_b15/t0 Drive'da duruyor.
import os
TANI_B = "/content/kod/deneme2/model_b/tani_b.py"
EV_B6 = os.path.dirname(EV) + "/model_b15"
assert os.path.isdir(f"{EV_B6}/t0"), f"model_b15/t0 YOK: {EV_B6}"
!python {TANI_B} {EV_B6}/t0 --genislik 5